# RouteOpt: graph-based route optimization
Shortest-path algorithms, graph traversal, Trie autocomplete and benchmarking on a weighted map.

**Run in Colab:** upload this file at colab.research.google.com (File > Upload notebook), then Runtime > Run all.
Everything below is generated by the code in the notebook, nothing is hard-coded.

In [ ]:
!pip -q install plotly ipywidgets

## 0. Core library
The cell below writes `routeopt.py` (graph, binary-heap Dijkstra and A*, Bellman-Ford, BFS/DFS, topological sort, Trie, benchmarks, plots).

In [ ]:
%%writefile routeopt.py
"""RouteOpt core: weighted graph, shortest-path algorithms, traversal, Trie and benchmarking."""
from __future__ import annotations

import heapq
import math
import random
import statistics
import time
import tracemalloc
from collections import deque
from dataclasses import dataclass

NAMES = ["Harbour Gate", "Mill Lane", "Old Market", "Canal Street", "Station Square", "Library",
         "Clock Tower", "Botanic Garden", "Riverside", "Fort Road", "Orchard Hill", "Chapel Yard",
         "Bridge End", "Foundry", "Granary", "Hospital", "Ironworks", "Junction", "Kiln Row",
         "Lantern Court", "Museum", "North Gate", "Observatory", "Parkside", "Quarry Bend",
         "Rose Court", "South Quay", "Tannery", "University", "Viaduct", "Wharf", "Yew Tree",
         "Zenith Plaza", "Abbey", "Brook End", "Cathedral",
         "Alder Gate", "Birch Crescent", "Cedar Wharf", "Dove Lane", "Elder Court", "Fern Hollow",
         "Grove End", "Hazel Drive", "Ivy Terrace", "Juniper Row", "Acorn Way", "Beechwood",
         "Chestnut Hill", "Dogwood", "Elm Park", "Fig Orchard", "Gorse Field", "Holly Bank",
         "Iris Meadow", "Jasmine Walk", "Kingfisher Quay", "Larkspur Lane", "Magnolia Court", "Nettlebed",
         "Oakenshaw", "Primrose Hill", "Quince Orchard", "Rowan Gate", "Sycamore", "Thistledown",
         "Umber Quay", "Vetch Field", "Willowbank", "Yarrow", "Amberley", "Brambletye",
         "Copperfield", "Dunmore", "Easthaven", "Foxglove", "Goldcrest", "Heathfield",
         "Inkwell", "Jubilee", "Kestrel Rise", "Linen Yard", "Marlowe", "Netherby",
         "Osprey Point", "Plover Court", "Quartz Row", "Ravenscar", "Saltmarsh", "Tidewater",
         "Upton", "Valve House", "Wexford", "Xylon", "Yarmouth", "Zennor",
         "Ashcombe", "Blackfriars", "Crows Nest", "Dovecote"]
TEAL, AMBER, ROUTE, INK, EDGE = "#0E7C86", "#E9A23B", "#C81E5A", "#14232D", "#B3C2CB"


# ---------------------------------------------------------------- graph
@dataclass
class Graph:
    pos: list   # (x, y) per node
    adj: list   # adjacency lists: adj[u] = [(v, weight), ...]

    @property
    def n(self):
        return len(self.pos)

    @property
    def edges(self):
        return [(u, v, w) for u in range(self.n) for v, w in self.adj[u] if u < v]

    @property
    def labels(self):
        return [NAMES[i] if i < len(NAMES) else f"Node {i}" for i in range(self.n)]


def make_grid_graph(rows=6, cols=6, seed=11, drop=0.25, diag=0.3, spread=0.6) -> Graph:
    """Jittered grid map. Weights are 0.8x to 1.4x the straight-line distance, so
    0.8 * Euclidean distance is an admissible, consistent A* heuristic."""
    rng = random.Random(seed)
    pos = [(c + rng.uniform(-.38, .38), r + rng.uniform(-.45, .45)) for r in range(rows) for c in range(cols)]
    adj = [[] for _ in pos]

    def link(a, b):
        w = math.dist(pos[a], pos[b]) * rng.uniform(.8, .8 + spread)
        adj[a].append((b, w))
        adj[b].append((a, w))

    for i in range(rows * cols):
        r, c = divmod(i, cols)
        if c < cols - 1 and (r == 0 or rng.random() > drop):
            link(i, i + 1)
        if r < rows - 1:
            link(i, i + cols)          # all vertical edges + top row keep the graph connected
        if r < rows - 1 and c < cols - 1 and rng.random() < diag:
            link(i, i + cols + 1)
        if r < rows - 1 and c > 0 and rng.random() < diag * .67:
            link(i, i + cols - 1)
    return Graph(pos, adj)


# ---------------------------------------------------------------- shortest paths
@dataclass
class Result:
    name: str
    path: list
    cost: float
    visited: list          # nodes in the order the search touched them
    edges_checked: int
    seconds: float
    pushes: int = 0        # heap pushes (Dijkstra, A*)
    peak_heap: int = 0     # largest heap size reached
    passes: int = 0        # Bellman-Ford passes over all edges

    @property
    def found(self):
        return bool(self.path)


def _finish(name, g, order, edges, prev, s, t, t0):
    secs = time.perf_counter() - t0
    path, v = [], t
    while v != -1:
        path.append(v)
        if v == s:
            break
        v = prev[v]
    path.reverse()
    if not path or path[0] != s:
        path = []
    cost = sum(dict(g.adj[a])[b] for a, b in zip(path, path[1:]))
    return Result(name, path, cost, order, edges, secs)


def _best_first(g, s, t, use_h, name):
    t0 = time.perf_counter()
    h = (lambda n: 0.8 * math.dist(g.pos[n], g.pos[t])) if use_h else (lambda n: 0.0)
    dist, prev = [math.inf] * g.n, [-1] * g.n
    dist[s], done, order, edges = 0.0, set(), [], 0
    heap = [(h(s), s)]                       # binary heap (heapq)
    pushes, peak = 1, 1
    while heap:
        _, u = heapq.heappop(heap)
        if u in done:
            continue
        done.add(u)
        order.append(u)
        if u == t:
            break
        for v, w in g.adj[u]:
            edges += 1
            nd = dist[u] + w
            if nd < dist[v]:
                dist[v], prev[v] = nd, u
                heapq.heappush(heap, (nd + h(v), v))
                pushes += 1
                peak = max(peak, len(heap))
    r = _finish(name, g, order, edges, prev, s, t, t0)
    r.pushes, r.peak_heap = pushes, peak
    return r


def dijkstra(g, s, t):
    return _best_first(g, s, t, False, "Dijkstra")


def astar(g, s, t):
    return _best_first(g, s, t, True, "A*")


def bellman_ford(g, s, t):
    t0 = time.perf_counter()
    dist, prev = [math.inf] * g.n, [-1] * g.n
    dist[s], seen, order, edges = 0.0, {s}, [s], 0
    passes = 0
    for _ in range(g.n - 1):
        passes += 1
        changed = False
        for u in range(g.n):
            if dist[u] == math.inf:
                continue
            for v, w in g.adj[u]:
                edges += 1
                if dist[u] + w < dist[v]:
                    dist[v], prev[v], changed = dist[u] + w, u, True
                    if v not in seen:
                        seen.add(v)
                        order.append(v)
        if not changed:
            break
    r = _finish("Bellman-Ford", g, order, edges, prev, s, t, t0)
    r.passes = passes
    return r


def bfs(g, s, t):
    """Fewest hops, ignoring weights."""
    t0 = time.perf_counter()
    prev, seen, q, order, edges = [-1] * g.n, {s}, deque([s]), [], 0
    while q:
        u = q.popleft()
        order.append(u)
        if u == t:
            break
        for v, _ in g.adj[u]:
            edges += 1
            if v not in seen:
                seen.add(v)
                prev[v] = u
                q.append(v)
    return _finish("BFS", g, order, edges, prev, s, t, t0)


ALGORITHMS = {"Dijkstra": dijkstra, "A*": astar, "Bellman-Ford": bellman_ford, "BFS": bfs}


# ---------------------------------------------------------------- traversal, DAGs
def bfs_order(g, s):
    seen, q, out = {s}, deque([s]), []
    while q:
        u = q.popleft()
        out.append(u)
        for v, _ in g.adj[u]:
            if v not in seen:
                seen.add(v)
                q.append(v)
    return out


def dfs_order(g, s):
    seen, stack, out = set(), [s], []
    while stack:
        u = stack.pop()
        if u in seen:
            continue
        seen.add(u)
        out.append(u)
        stack.extend(v for v, _ in reversed(g.adj[u]) if v not in seen)
    return out


def components(g):
    seen, comps = set(), []
    for s in range(g.n):
        if s not in seen:
            comp = bfs_order(g, s)
            seen.update(comp)
            comps.append(comp)
    return comps


def topological_sort(deps: dict):
    """deps maps task -> list of prerequisites. Kahn's algorithm; raises ValueError on a cycle."""
    nodes = set(deps) | {p for ps in deps.values() for p in ps}
    waiting = {n: set(deps.get(n, [])) for n in nodes}
    children = {n: [] for n in nodes}
    for n, ps in waiting.items():
        for p in ps:
            children[p].append(n)
    ready, out = sorted(n for n, ps in waiting.items() if not ps), []
    while ready:
        n = ready.pop(0)
        out.append(n)
        for c in sorted(children[n]):
            waiting[c].discard(n)
            if not waiting[c]:
                ready.append(c)
    if len(out) < len(nodes):
        raise ValueError("cycle among: " + ", ".join(sorted(nodes - set(out))))
    return out


def has_cycle(deps: dict) -> bool:
    try:
        topological_sort(deps)
        return False
    except ValueError:
        return True


# ---------------------------------------------------------------- Trie autocomplete
class Trie:
    def __init__(self, words=()):
        self.root = {}
        for i, w in enumerate(words):
            self.insert(w, i)

    def insert(self, word, value):
        node = self.root
        for ch in word.lower():
            node = node.setdefault(ch, {})
        node.setdefault("$", []).append(value)

    def complete(self, prefix, limit=8):
        node = self.root
        for ch in prefix.lower():
            if ch not in node:
                return []
            node = node[ch]
        out = []

        def walk(n):
            if len(out) >= limit:
                return
            out.extend(n.get("$", []))
            for ch in sorted(k for k in n if k != "$"):
                walk(n[ch])

        walk(node)
        return out[:limit]


# ---------------------------------------------------------------- benchmarking
def benchmark(sides=(10, 20, 30, 40), repeats=3, seed=5):
    """Corner-to-corner searches on k x k grids: median runtime, nodes visited, peak memory."""
    rows = []
    for k in sides:
        g = make_grid_graph(k, k, seed)
        s, t = 0, g.n - 1
        for name, fn in ALGORITHMS.items():
            runs = [fn(g, s, t) for _ in range(repeats)]
            tracemalloc.start()
            fn(g, s, t)
            peak = tracemalloc.get_traced_memory()[1]
            tracemalloc.stop()
            r = runs[-1]
            rows.append(dict(nodes=g.n, algorithm=name, ms=1000 * statistics.median(x.seconds for x in runs),
                             visited=len(r.visited), edges_checked=r.edges_checked,
                             peak_kb=peak / 1024, cost=r.cost))
    return rows


def pair_effort(g, n_pairs=100, seed=1):
    """Run every algorithm on random start/destination pairs."""
    rng, rows = random.Random(seed), []
    for _ in range(n_pairs):
        s, t = rng.sample(range(g.n), 2)
        for name, fn in ALGORITHMS.items():
            r = fn(g, s, t)
            rows.append(dict(algorithm=name, visited=len(r.visited), edges_checked=r.edges_checked,
                             ms=1000 * r.seconds, cost=r.cost))
    return rows


def visit_counts(g, algo="Dijkstra", max_pairs=400, seed=1):
    """How often each location is visited when searching every pair (or a random sample of pairs)."""
    allp = [(s, t) for s in range(g.n) for t in range(s + 1, g.n)]
    pairs = allp if len(allp) <= max_pairs else random.Random(seed).sample(allp, max_pairs)
    counts, fn = [0] * g.n, ALGORITHMS[algo]
    for s, t in pairs:
        for v in fn(g, s, t).visited:
            counts[v] += 1
    return counts, len(pairs)


def optimality_gap(g, n_pairs=200, seed=1):
    """Average % by which each algorithm's route costs more than the cheapest route."""
    rng, tot = random.Random(seed), {k: 0.0 for k in ALGORITHMS}
    for _ in range(n_pairs):
        s, t = rng.sample(range(g.n), 2)
        base = dijkstra(g, s, t).cost
        for k, fn in ALGORITHMS.items():
            tot[k] += fn(g, s, t).cost / base
    return {k: 100 * (v / n_pairs - 1) for k, v in tot.items()}


def effort_by_hops(g, max_pairs=500, seed=1):
    """Average nodes visited, grouped by the hop distance between start and destination."""
    allp = [(s, t) for s in range(g.n) for t in range(s + 1, g.n)]
    pairs = allp if len(allp) <= max_pairs else random.Random(seed).sample(allp, max_pairs)
    acc = {}
    for s, t in pairs:
        hops = len(bfs(g, s, t).path) - 1
        for name, fn in ALGORITHMS.items():
            acc.setdefault((hops, name), []).append(len(fn(g, s, t).visited))
    return [dict(hops=h, algorithm=a, visited=sum(v) / len(v), pairs=len(v)) for (h, a), v in sorted(acc.items())]


def astar_savings(g, n_pairs=300, seed=1):
    """For random pairs: % fewer nodes A* visits than Dijkstra."""
    rng, out = random.Random(seed), []
    for _ in range(n_pairs):
        s, t = rng.sample(range(g.n), 2)
        out.append(100 * (1 - len(astar(g, s, t).visited) / len(dijkstra(g, s, t).visited)))
    return out


def heap_stats(g, n_pairs=200, seed=1):
    """Average heap pushes and peak heap size for Dijkstra and A*."""
    rng, rows = random.Random(seed), {"Dijkstra": [0, 0], "A*": [0, 0]}
    for _ in range(n_pairs):
        s, t = rng.sample(range(g.n), 2)
        for name, fn in (("Dijkstra", dijkstra), ("A*", astar)):
            r = fn(g, s, t)
            rows[name][0] += r.pushes
            rows[name][1] += r.peak_heap
    return [dict(algorithm=k, avg_pushes=v[0] / n_pairs, avg_peak_heap=v[1] / n_pairs) for k, v in rows.items()]


def cost_from(g, s):
    """Cheapest route cost from s to every node (Dijkstra without a target)."""
    dist, heap = [math.inf] * g.n, [(0.0, s)]
    dist[s] = 0.0
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]:
            continue
        for v, w in g.adj[u]:
            if d + w < dist[v]:
                dist[v] = d + w
                heapq.heappush(heap, (dist[v], v))
    return dist


def route_stats(g, max_pairs=600, seed=1):
    """Cheapest routes for many pairs: how often each road is used, plus cost, hops and detour per pair."""
    allp = [(s, t) for s in range(g.n) for t in range(s + 1, g.n)]
    pairs = allp if len(allp) <= max_pairs else random.Random(seed).sample(allp, max_pairs)
    usage, rows = {}, []
    for s, t in pairs:
        r = dijkstra(g, s, t)
        for a, b in zip(r.path, r.path[1:]):
            usage[(min(a, b), max(a, b))] = usage.get((min(a, b), max(a, b)), 0) + 1
        straight = math.dist(g.pos[s], g.pos[t])
        rows.append(dict(cost=r.cost, hops=len(r.path) - 1, straight=straight, detour=r.cost / straight))
    return usage, rows


def betweenness(g, max_pairs=600, seed=1):
    """How many cheapest routes pass through each location (as an intermediate stop)."""
    allp = [(s, t) for s in range(g.n) for t in range(s + 1, g.n)]
    pairs = allp if len(allp) <= max_pairs else random.Random(seed).sample(allp, max_pairs)
    counts = [0] * g.n
    for s, t in pairs:
        for v in dijkstra(g, s, t).path[1:-1]:
            counts[v] += 1
    return counts


def bf_passes(g, max_pairs=300, seed=1):
    """Passes Bellman-Ford needs before nothing improves, for many pairs."""
    allp = [(s, t) for s in range(g.n) for t in range(s + 1, g.n)]
    pairs = allp if len(allp) <= max_pairs else random.Random(seed).sample(allp, max_pairs)
    return [bellman_ford(g, s, t).passes for s, t in pairs]


def weight_sensitivity(spreads=(0, .3, .6, .9, 1.2), rows=6, cols=6, seed=11, n_pairs=300):
    """Rebuild the map with wider road-weight ranges (0.8 to 0.8+spread times distance)."""
    out = []
    for sp in spreads:
        g = make_grid_graph(rows, cols, seed, spread=sp)
        rng, gap, sav = random.Random(1), 0.0, 0.0
        for _ in range(n_pairs):
            s, t = rng.sample(range(g.n), 2)
            d = dijkstra(g, s, t)
            gap += bfs(g, s, t).cost / d.cost
            sav += 1 - len(astar(g, s, t).visited) / len(d.visited)
        out.append(dict(spread=sp, bfs_extra_cost_pct=100 * (gap / n_pairs - 1), astar_fewer_nodes_pct=100 * sav / n_pairs))
    return out


# ---------------------------------------------------------------- plotting (plotly)
def _base(g, height):
    import plotly.graph_objects as go
    X, Y = [p[0] for p in g.pos], [-p[1] for p in g.pos]
    ex, ey = [], []
    for u, v, _ in g.edges:
        ex += [X[u], X[v], None]
        ey += [Y[u], Y[v], None]
    fig = go.Figure(go.Scatter(x=ex, y=ey, mode="lines", line=dict(color=EDGE, width=1.5), hoverinfo="skip"))
    fig.update_layout(template="plotly_white", showlegend=False, height=height,
                      margin=dict(l=8, r=8, t=40, b=8),
                      xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x"))
    return fig, X, Y


def plot_graph(g, start=None, end=None, visited=(), path=(), title="", height=520):
    import plotly.graph_objects as go
    fig, X, Y = _base(g, height)
    if len(path) > 1:
        fig.add_trace(go.Scatter(x=[X[i] for i in path], y=[Y[i] for i in path], mode="lines",
                                 line=dict(color=ROUTE, width=5), hoverinfo="skip"))
    colors = ["#FFFFFF"] * g.n
    for i in visited:
        colors[i] = AMBER
    for i in path:
        colors[i] = ROUTE
    if start is not None:
        colors[start] = TEAL
    if end is not None:
        colors[end] = INK
    size = max(5, min(14, 22 - 2 * math.sqrt(g.n)))
    fig.add_trace(go.Scatter(x=X, y=Y, mode="markers", text=g.labels, hovertemplate="%{text}<extra></extra>",
                             marker=dict(size=size, color=colors, line=dict(color=INK, width=1.5))))
    ends = [i for i in (start, end) if i is not None]
    fig.add_trace(go.Scatter(x=[X[i] for i in ends], y=[Y[i] for i in ends], mode="text",
                             text=[g.labels[i] for i in ends], textposition="top center", hoverinfo="skip"))
    fig.update_layout(title=title)
    return fig


def plot_order(g, order, title="", height=420):
    """Colour every node by the rank at which a traversal reached it."""
    import plotly.graph_objects as go
    fig, X, Y = _base(g, height)
    rank = [None] * g.n
    for k, i in enumerate(order):
        rank[i] = k
    fig.add_trace(go.Scatter(
        x=X, y=Y, mode="markers", text=[f"{l}: step {r}" for l, r in zip(g.labels, rank)],
        hovertemplate="%{text}<extra></extra>",
        marker=dict(size=12, color=[r if r is not None else -1 for r in rank], colorscale="Tealgrn",
                    showscale=True, colorbar=dict(title="step"), line=dict(color=INK, width=1))))
    fig.update_layout(title=title)
    return fig


def plot_heat(g, counts, title="", height=460):
    """Shade each node by a count (for example how often searches visit it)."""
    import plotly.graph_objects as go
    fig, X, Y = _base(g, height)
    fig.add_trace(go.Scatter(
        x=X, y=Y, mode="markers", text=[f"{l}: {c}" for l, c in zip(g.labels, counts)],
        hovertemplate="%{text}<extra></extra>",
        marker=dict(size=14, color=counts, colorscale="Tealgrn", showscale=True,
                    colorbar=dict(title="visits"), line=dict(color=INK, width=1.5))))
    fig.update_layout(title=title)
    return fig


def plot_edge_usage(g, usage, title="", height=460):
    """Road thickness shows how many cheapest routes use each road."""
    import plotly.graph_objects as go
    fig, X, Y = _base(g, height)
    mx, labels = max(usage.values()), g.labels
    for (a, b), c in usage.items():
        fig.add_trace(go.Scatter(x=[X[a], X[b]], y=[Y[a], Y[b]], mode="lines", hoverinfo="skip",
                                 line=dict(color=TEAL, width=1 + 9 * c / mx)))
    mids = list(usage)
    fig.add_trace(go.Scatter(x=[(X[a] + X[b]) / 2 for a, b in mids], y=[(Y[a] + Y[b]) / 2 for a, b in mids],
                             mode="markers", marker=dict(size=8, opacity=0),
                             text=[f"{labels[a]} to {labels[b]}: {usage[(a, b)]} routes" for a, b in mids],
                             hovertemplate="%{text}<extra></extra>"))
    fig.add_trace(go.Scatter(x=X, y=Y, mode="markers", hoverinfo="skip",
                             marker=dict(size=6, color="#FFFFFF", line=dict(color=INK, width=1.5))))
    fig.update_layout(title=title)
    return fig


In [ ]:
import numpy as np, pandas as pd, plotly.express as px
import ipywidgets as widgets
import routeopt as ro
from IPython.display import display

## 1. The map
Locations are vertices and roads are weighted edges, stored as adjacency lists. Each weight is 0.8x to 1.4x the straight-line distance.

In [ ]:
g = ro.make_grid_graph(6, 6, seed=11)
print(g.n, 'locations,', len(g.edges), 'roads')
ro.plot_graph(g, title='The map').show()

## 2. Explore one search
Pick an algorithm, a start and a destination, then drag the step slider to replay the order in which nodes were visited.

In [ ]:
names = [(l, i) for i, l in enumerate(g.labels)]
def explore(algo, start, end, step):
    res = ro.ALGORITHMS[algo](g, start, end)
    step = min(step, len(res.visited))
    done = step == len(res.visited)
    print(f"visited {len(res.visited)} nodes | edges checked {res.edges_checked} | cost {res.cost:.2f} | stops {len(res.path)}")
    ro.plot_graph(g, start, end, res.visited[:step], res.path if done else (), title=f"{algo}: {g.labels[start]} to {g.labels[end]}").show()
widgets.interact(explore, algo=list(ro.ALGORITHMS), start=widgets.Dropdown(options=names, value=0),
                 end=widgets.Dropdown(options=names, value=g.n - 1), step=widgets.IntSlider(value=g.n, min=0, max=g.n, continuous_update=False));

## 3. Compare all four on one pair

In [ ]:
rs = [fn(g, 0, g.n - 1) for fn in ro.ALGORITHMS.values()]
df = pd.DataFrame([dict(algorithm=r.name, visited=len(r.visited), edges_checked=r.edges_checked,
                        cost=round(r.cost, 2), ms=round(r.seconds * 1000, 3), hops=len(r.path) - 1) for r in rs])
display(df)
px.bar(df.melt('algorithm', ['visited', 'edges_checked']), x='algorithm', y='value', color='variable', barmode='group',
       title='Search effort, corner to corner').show()

## 4. Average over random pairs
One pair can mislead. This runs every algorithm on many random start/destination pairs.

In [ ]:
ef = pd.DataFrame(ro.pair_effort(g, n_pairs=200))
display(ef.groupby('algorithm').mean().round(3))
px.box(ef, x='algorithm', y='visited', points='all', title='Nodes visited over 200 random pairs').show()
px.box(ef, x='algorithm', y='ms', title='Runtime (ms) over 200 random pairs').show()

## 5. Scaling benchmark
Corner-to-corner searches on k x k grids. Runtime is the median of repeated runs, memory is the peak from `tracemalloc`. The slope on a log-log plot estimates the empirical growth exponent.

In [ ]:
bd = pd.DataFrame(ro.benchmark(sides=(10, 20, 30, 40), repeats=3))
display(bd.round(3))

In [ ]:
px.line(bd, x='nodes', y='ms', color='algorithm', markers=True, log_x=True, log_y=True, title='Runtime (ms, log-log)').show()
px.line(bd, x='nodes', y='visited', color='algorithm', markers=True, log_x=True, log_y=True, title='Nodes visited (log-log)').show()
px.line(bd, x='nodes', y='peak_kb', color='algorithm', markers=True, log_x=True, title='Peak memory (KiB)').show()
slopes = {a: np.polyfit(np.log(s.nodes), np.log(s.ms), 1)[0] for a, s in bd.groupby('algorithm')}
pd.DataFrame({'empirical exponent': slopes}).round(2)

## 6. Trie autocomplete
Prefix search over location names. Lookup cost depends on the prefix length, not on how many locations exist.

In [ ]:
trie = ro.Trie(g.labels)
def complete(prefix=''):
    print([g.labels[i] for i in trie.complete(prefix)] if prefix else 'type a prefix')
widgets.interact(complete, prefix=widgets.Text(value='Ca', description='Prefix'));

## 7. Traversal, connectivity and dependency ordering
BFS reaches nodes ring by ring, DFS goes deep first. Topological sort orders tasks so every prerequisite comes first, and a cycle makes that impossible.

In [ ]:
ro.plot_order(g, ro.bfs_order(g, 0), 'BFS from the first location').show()
ro.plot_order(g, ro.dfs_order(g, 0), 'DFS from the first location').show()
print('connected components:', len(ro.components(g)))

In [ ]:
deps = {'deploy': ['test', 'build'], 'test': ['build'], 'build': ['fetch', 'configure']}
print('order:', ro.topological_sort(deps))
cyclic = {**deps, 'fetch': ['deploy']}
print('has cycle after adding fetch -> deploy:', ro.has_cycle(cyclic))

## 6b. Map analytics
Which locations do searches visit most, how much does each route cost compared with the best one, and what does the map look like in numbers?

In [ ]:
counts, npairs = ro.visit_counts(g, 'Dijkstra')
ro.plot_heat(g, counts, f'Visits per location across {npairs} searches (Dijkstra)').show()
gap = ro.optimality_gap(g)
px.bar(x=list(gap), y=list(gap.values()), labels=dict(x='algorithm', y='% over cheapest route'),
       title='Route quality: average extra cost over the cheapest route').show()

In [ ]:
ed = g.edges
print(f"{g.n} locations, {len(ed)} roads, {2 * len(ed) / g.n:.1f} roads per location, {len(ro.components(g))} component(s)")
px.histogram(x=[len(a) for a in g.adj], labels=dict(x='roads at a location'), title='Roads per location').show()
px.histogram(x=[w for _, _, w in ed], nbins=12, labels=dict(x='road weight'), title='Road weight').show()

## 6c. More data
How effort changes with distance, how much A* saves over Dijkstra, and what the binary heap does.

In [ ]:
bh = pd.DataFrame(ro.effort_by_hops(g))
px.line(bh, x='hops', y='visited', color='algorithm', markers=True, title='Average nodes visited by hop distance').show()
sv = ro.astar_savings(g)
print(f"A* visits {np.mean(sv):.0f}% fewer nodes than Dijkstra on average")
px.histogram(x=sv, nbins=10, labels=dict(x='% fewer nodes visited than Dijkstra'), title='A* savings over 300 random pairs').show()
hs = pd.DataFrame(ro.heap_stats(g)); display(hs.round(1))
px.bar(hs.melt('algorithm'), x='algorithm', y='value', color='variable', barmode='group', title='Priority queue activity').show()

## 6d. Distances, road usage and route costs
How far is everything from one place, which roads carry the most cheapest routes, and how winding are the routes?

In [ ]:
cf = ro.cost_from(g, 0)
ro.plot_heat(g, [round(v, 1) for v in cf], f'Cheapest route cost from {g.labels[0]}').show()
usage, rows_ = ro.route_stats(g)
ro.plot_edge_usage(g, usage, f'Roads used by {len(rows_)} cheapest routes').show()
print(len(g.edges) - len(usage), 'roads are on no cheapest route')

In [ ]:
rdf = pd.DataFrame(rows_)
px.histogram(rdf, x='cost', nbins=12, title='Cheapest route cost, all pairs').show()
px.histogram(rdf, x='detour', nbins=12, title='Route cost / straight-line distance (1.0 = straight road)').show()
px.scatter(rdf, x='hops', y='cost', opacity=0.5, title='Cost against hops for every pair').show()

## 6e. Central locations, Bellman-Ford passes and weight sensitivity

In [ ]:
bt = ro.betweenness(g)
ro.plot_heat(g, bt, 'Cheapest routes passing through each location').show()
px.histogram(x=ro.bf_passes(g), labels=dict(x='passes until nothing improves'), title='Bellman-Ford passes').show()
ws = pd.DataFrame(ro.weight_sensitivity()); display(ws.round(2))
px.line(ws.melt('spread', var_name='measure', value_name='percent'), x='spread', y='percent', color='measure', markers=True,
        title='What if road weights vary more? (weights = 0.8 to 0.8+spread times distance)').show()

## 6f. Export the datasets
Write the same CSVs the website serves from `data/`, so every chart is reproducible.

In [ ]:
import csv, os
os.makedirs('data', exist_ok=True)
def write_map(g, nodes_fn, edges_fn):
    with open(nodes_fn, 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['id', 'label', 'x', 'y'])
        for i, (x, y) in enumerate(g.pos):
            w.writerow([i, g.labels[i], round(x, 4), round(y, 4)])
    with open(edges_fn, 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['u', 'v', 'u_label', 'v_label', 'weight'])
        for u, v, wt in g.edges:
            w.writerow([u, v, g.labels[u], g.labels[v], round(wt, 4)])
write_map(g, 'data/nodes.csv', 'data/edges.csv')
g8 = ro.make_grid_graph(8, 8, seed=11)
write_map(g8, 'data/nodes_8x8.csv', 'data/edges_8x8.csv')
def write_bench(g, fn, n):
    with open(fn, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['algorithm', 'visited', 'edges_checked', 'ms', 'cost'])
        w.writeheader()
        for r in ro.pair_effort(g, n_pairs=n):
            w.writerow({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()})
write_bench(g, 'data/benchmark_allpairs.csv', 100)
write_bench(g8, 'data/benchmark_8x8.csv', 300)
with open('data/scaling_benchmark.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['nodes', 'algorithm', 'ms', 'visited', 'edges_checked', 'peak_kb', 'cost'])
    w.writeheader()
    for r in ro.benchmark(sides=(5, 10, 15, 20, 30), repeats=3):
        w.writerow({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()})
print('data/ written:', sorted(os.listdir('data')))


## What to look for
* **A\*** should visit fewer nodes than Dijkstra for the same route cost, because the heuristic points it at the destination.
* **Bellman-Ford** checks far more edges: it repeats passes over every edge, O(V·E) in the worst case, which is the price for tolerating negative weights.
* **BFS** ignores weights, so it finds the fewest hops, which is not always the cheapest route.
* The growth exponents in section 5 are measured on this machine and this graph family, so treat them as estimates.